# Forza steering training
Choose a GPU runtime. Clone the private repository using your preferred authenticated method, then set `REPO`. Never save a token in this notebook. The notebook delegates all training logic to the package.


In [ ]:
from pathlib import Path
REPO = Path('/content/stormhacks')
assert (REPO / 'pyproject.toml').exists(), 'Clone the repository first'
%cd {REPO}
%pip install -e '.[training]'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Place completed session archives on Drive. Extract **trusted** archives into `/content/forza-data`, with session folders directly under it, before running validation. Avoid image reads over Drive during training. Each metadata file must declare `completed: true`. Checkpoints below persist on Drive; a new run requires an empty output folder.


In [ ]:
import torch
DATA = Path('/content/forza-data')
RUN = Path('/content/drive/MyDrive/forza-ai/runs/baseline-001')
assert torch.cuda.is_available(), 'Select a GPU runtime for real training'
print(torch.cuda.get_device_name(0))


In [ ]:
from forza_ai.training.cli import main
main(['validate', str(DATA)])


In [ ]:
main(['train', str(DATA), str(RUN), '--epochs', '10', '--device', 'cuda', '--batch-size', '64'])


After interruption, rerun setup and data extraction, skip the new-run training cell, then use resume. Epoch count is the total desired count. Resume verifies the dataset fingerprints and restores configuration, split, model and optimizer.


In [ ]:
# Uncomment to resume to 20 total epochs:
# main(['resume', str(RUN / 'last.pt'), str(DATA), '--epochs', '20', '--device', 'cuda'])


In [ ]:
main(['evaluate', str(RUN / 'best.pt'), str(DATA), '--device', 'cuda'])
EXPORT = RUN / 'export-best'
main(['export', str(RUN / 'best.pt'), str(EXPORT)])


Download the exported directory for CPU shadow inference. Good offline errors do not establish driving quality. Before real data arrives, use `forza-train synthetic` plus the documented CPU smoke workflow in `docs/TRAINING.md`.
